# Lab: Bayes learning and multi-armed bandits

**Due:** before class on Wednesday 18 November 2026.

**Submit:** on Canvas, upload this notebook as one `.ipynb` file. Before you submit, choose Kernel, then Restart and Run All, and confirm that every check cell prints `ok`.

**How the checks work.** Each exercise ends with a check cell.

- `ok  name` means your value matches the reference.
- `not yet: ...` (an error) means it does not match. Re-read the task, the inputs and the rounding, then run again.
- Exercises with a simulation (marked MC) accept a small tolerance, stated in the task. Use your own `np.random.default_rng(seed)`.
- Run All stops at the first empty exercise. That is expected. Work in order.

Write your code in the empty cell under each task. Define the variables named in the task. Do not edit the check cells.

| # | Topic | Slide topic |
|---|---|---|
| 1 | Beta posterior, mean and 95% interval (SimCity) | Beta-binomial updating |
| 2 | Posterior predictive for black swans (MC) | Posterior predictive |
| 3 | Gamma posterior for an event rate | Poisson-Gamma conjugate update |
| 4 | Bears shrinkage and predictive probability | Normal-normal shrinkage |
| 5 | Which arm is better, by Monte Carlo (MC) | P(A better than B) |
| 6 | Thompson sampling with a fixed seed (MC) | Thompson sampling, explore and exploit |
| 7 | Q <- Q + alpha (r - Q), alpha = 1/n against constant alpha | Incremental update |


In [ ]:
from hashlib import sha256

import numpy as np
from scipy import stats

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: replace None for {name}")
    try:
        s = f"{round(float(value), ndigits):.{ndigits}f}"
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} must be a single number, not {type(value).__name__}")
    if not np.isfinite(float(value)):
        raise AssertionError(f"not yet: {name} is not finite (nan or inf)")
    got = sha256(f"aii600-bayes-bandits|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} does not match the reference value (you got {s}); check the formula and the inputs"
    print("ok ", name)

def check_array(name, value, expected, shape, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: replace None for {name}")
    value = np.asarray(value, dtype=float)
    if value.shape != shape:
        raise AssertionError(f"not yet: {name} must have shape {shape}, but has shape {value.shape}")
    if not np.all(np.isfinite(value)):
        raise AssertionError(f"not yet: {name} contains nan or inf")
    s = ",".join(f"{round(float(v), ndigits):.{ndigits}f}" for v in value.ravel())
    got = sha256(f"aii600-bayes-bandits|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} has the right shape but the values do not match the reference; check each update"
    print("ok ", name)

def check_close(name, value, accepted, width):
    # simulation check: passes when the value is within about `width` of the reference
    if value is None:
        raise AssertionError(f"not yet: replace None for {name}")
    try:
        v = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} must be a single number, not {type(value).__name__}")
    if not np.isfinite(v):
        raise AssertionError(f"not yet: {name} is not finite (nan or inf)")
    k = int(round(v / width))
    got = sha256(f"aii600-bayes-bandits|{name}|bin{k}".encode()).hexdigest()[:16]
    assert got in accepted, f"not yet: {name} is not within {width} of the reference (you got {v:.4f}); check the formula, the inputs and the number of draws"
    print("ok ", name)


### Exercise 1: Beta posterior (SimCity)

Task: find the posterior of the pink page's purchase rate from a flat prior, its mean, and its 95% equal-tailed credible interval.

Inputs: `buys_pink = 200` buys among `n_pink = 500` visitors (book SimCity data). Prior Beta(`a0`, `b0`) = Beta(1, 1).

Formulas: the posterior is Beta(`post_a`, `post_b`) with `post_a = a0 + buys_pink` and `post_b = b0 + n_pink - buys_pink`. The mean is `post_a / (post_a + post_b)`. The interval is the 2.5% and 97.5% quantiles of the posterior (`stats.beta.ppf`).

Define: `post_a`, `post_b`, `post_mean`, `ci_lo`, `ci_hi`. Rates as numbers between 0 and 1, no rounding.


In [ ]:
buys_pink, n_pink = 200, 500
a0, b0 = 1, 1


In [ ]:
# Exercise 1: define post_a, post_b, post_mean, ci_lo, ci_hi


In [ ]:
check("post_a", post_a, "a5dd147257132ffd")
check("post_b", post_b, "aa703b16d6fe1d26")
check("post_mean", post_mean, "20d57acff0c1d51b")
check("ci_lo", ci_lo, "7d43c2af41351347")
check("ci_hi", ci_hi, "fb3eebfa7a45a259")
print("ok  Exercise 1")


### Exercise 2: Posterior predictive, black swans (MC)

Task: after 100 white swans and no black swan, find the probability that the next swan is white, and the probability of at least one black swan among the next 100.

Inputs: `T_swans = 100` swans seen, `Y_white = 100` of them white. Prior Beta(1, 1).

Formulas: the posterior is Beta(`al_n`, `be_n`) with `al_n = 1 + Y_white` and `be_n = 1 + T_swans - Y_white`. The next swan is white with probability `pred_white = al_n / (al_n + be_n)`. For the next 100 swans, `p_black100 = E[1 - theta^100]` under the posterior. Estimate it by Monte Carlo: draw at least 100,000 values of `theta` from the posterior and average `1 - theta**100`.

Define: `al_n`, `be_n`, `pred_white` (exact formula), `p_black100` (MC, tolerance 0.01).


In [ ]:
T_swans, Y_white = 100, 100


In [ ]:
# Exercise 2: define al_n, be_n, pred_white, p_black100


In [ ]:
check("al_n", al_n, "2d705419153696b6")
check("be_n", be_n, "45b63b77232a7467")
check("pred_white", pred_white, "a1162067d855a3bc")
check_close("p_black100", p_black100, ['1524e3e854e46538', '08c2ba4dbb636b16', '8419622a66ed4de7'], 0.01)
print("ok  Exercise 2")


### Exercise 3: Gamma posterior for an event rate

Task: update a Gamma prior for an event rate with counts, and report the posterior mean and a 90% interval.

Inputs: events per year, rate `lam`. Prior Gamma(`a_prior`, `A_prior`) = Gamma(4, 1), the high-activity prior of the book (mean 4 events per year, 5% to 95% prior range 1.37 to 7.75). Data: `N_events = 4` events observed over `t_years = 4` years.

Formulas: the posterior is Gamma(`a_t`, `A_t`) with `a_t = a_prior + N_events` (shape) and `A_t = A_prior + t_years` (rate). The mean is `a_t / A_t`. The interval is the 5% and 95% quantiles, `stats.gamma.ppf(q, a_t, scale=1 / A_t)`. Check your call on the prior first: it should return 1.37 and 7.75.

Define: `a_t`, `A_t`, `lam_mean`, `lam_lo`, `lam_hi`, in events per year.


In [ ]:
a_prior, A_prior = 4, 1
N_events, t_years = 4, 4


In [ ]:
# Exercise 3: define a_t, A_t, lam_mean, lam_lo, lam_hi


In [ ]:
check("a_t", a_t, "cdda66c9278c0b1b")
check("A_t", A_t, "62610c4656fba083")
check("lam_mean", lam_mean, "d237b3cae25b942a")
check("lam_lo", lam_lo, "0589a3d66d0f95ff")
check("lam_hi", lam_hi, "6d0b95b94e948054")
print("ok  Exercise 3")


### Exercise 4: Bears shrinkage and predictive probability

Task: shrink the Bears' average margin toward an average team, and find the probability that the next game beats the betting line.

Inputs: `margins` (points, own score minus opponent score, 9 games, book data), prior mean `mu0 = 0` and prior standard deviation `tau = 4` points, betting line `line = -3.5` points. Treat the standard deviation of one margin as known: `sigma = s`, the sample standard deviation of `margins` (`ddof=1`).

Formulas, with `n_games = 9` and `ybar` the sample mean:

- `B = tau**2 / (tau**2 + sigma**2 / n_games)`, the weight on the data.
- `mu_n = mu0 + B * (ybar - mu0)`, the posterior mean.
- `post_sd = sqrt(1 / (1 / tau**2 + n_games / sigma**2))`, the posterior standard deviation of the average margin.
- Predictive: the next margin is normal with mean `mu_n` and standard deviation `sqrt(post_sd**2 + sigma**2)`. `p_beat_line` is the probability that it is above `line`.

Define: `B`, `mu_n`, `post_sd`, `p_beat_line`. Points for `mu_n` and `post_sd`, a probability for `p_beat_line`.


In [ ]:
margins = np.array([-3, 8, 8, -21, -7, 14, -13, -28, -41])
mu0, tau, line = 0.0, 4.0, -3.5


In [ ]:
# Exercise 4: define B, mu_n, post_sd, p_beat_line


In [ ]:
check("B", B, "2ed8ac774c26eed8")
check("mu_n", mu_n, "5982280dad08560b")
check("post_sd", post_sd, "d98a47669d68a7ec")
check("p_beat_line", p_beat_line, "fe9c99a748e1c211")
print("ok  Exercise 4")


### Exercise 5: Which arm is better (MC)

Task: from three arms' counts, find the probability that arm 3 beats arm 2, and the probability that arm 3 is the best of the three.

Inputs (book three-arm example): `sessions = [20, 30, 40]`, `conversions = [12, 20, 30]`. Prior Beta(1, 1) for each arm.

Formulas: arm `k` has posterior Beta(`1 + conversions[k]`, `1 + sessions[k] - conversions[k]`). Draw at least 100,000 values from each posterior with your own `np.random.default_rng(seed)`. `p_3_over_2` is the share of draws with arm 3 above arm 2. `p_3_best` is the share of draws with arm 3 above both of the others.

Define: `p_3_over_2`, `p_3_best`, probabilities between 0 and 1 (MC, tolerance 0.01).


In [ ]:
sessions = np.array([20, 30, 40])
conversions = np.array([12, 20, 30])


In [ ]:
# Exercise 5: define p_3_over_2, p_3_best


In [ ]:
check_close("p_3_over_2", p_3_over_2, ['5d4b8336a8105390', 'b01131d8e11f613d', 'db05d3d6843371a6'], 0.01)
check_close("p_3_best", p_3_best, ['dd176faf6654eeeb', '25a82a8551271556', 'df17014310545055'], 0.01)
print("ok  Exercise 5")


### Exercise 6: Thompson sampling with a fixed seed (MC)

Task: run Thompson sampling on two pages and find the share of visitors shown the better page.

Inputs (book bandit example): `rates = [0.04, 0.05]` are the true conversion rates of arm 0 (original) and arm 1 (variation, the better arm). The algorithm does not know them. Run `R = 200` independent experiments of `T = 3000` visitors each, in parallel as arrays of shape `(R, 2)`.

Algorithm, for each of the `T` visitors:

1. Draw one value per arm and experiment from its Beta posterior, `rng.beta(a, b)`. Start with `a` and `b` equal to ones.
2. Show the arm with the larger draw (`argmax` over axis 1).
3. Draw the reward: a conversion with probability `rates[arm]` (`rng.random(R) < rates[arm]`).
4. Add the reward to `a` of the arm shown and `1 - reward` to its `b`.

Use your own `rng = np.random.default_rng(seed)`.

Define: `share_best`, the share of all `R * T` visitors who were shown arm 1. A 50/50 split gives 0.5. Tolerance 0.06.


In [ ]:
rates = np.array([0.04, 0.05])
R, T = 200, 3000


In [ ]:
# Exercise 6: define share_best


In [ ]:
check_close("share_best", share_best, ['71a00ce09552f3ba', '54604e0c49b365ae', '1fdf34fb18728e71'], 0.06)
print("ok  Exercise 6")


### Exercise 7: The incremental update, 1/n against constant alpha

Task: write the update `Q <- Q + alpha * (r - Q)` in a loop over logged visitors and compare two step sizes.

Inputs: `arm_log` (0 = pink, 1 = black) and `reward_log` (1 = buy, 0 = no buy) for 40 simulated visitors, from the cell above. Pink's true rate is 0.6 for the first 20 visitors and 0.2 after that. Keep one estimate per page, both starting at 0.

Formulas: `q_1n` uses `alpha = 1 / n`, where `n` counts the visitors shown that page so far, including this one. `q_const` uses the constant `alpha = 0.2`. For each visitor, update only the estimate of the page shown.

Define: `q_1n` and `q_const`, arrays of shape `(2,)` with the final estimates for (pink, black). No rounding.


In [ ]:
# Simulated log: 40 visitors, arm 0 = pink, arm 1 = black.
rng_q = np.random.default_rng(55)
T_q = 40
arm_log = rng_q.integers(0, 2, T_q)
rate_now = np.where(np.arange(T_q) < 20, 0.6, 0.2)
reward_log = (rng_q.random(T_q) < np.where(arm_log == 0, rate_now, 0.4)).astype(float)
print("visitors:", T_q, " shown pink:", int((arm_log == 0).sum()), " shown black:", int((arm_log == 1).sum()))


In [ ]:
# Exercise 7: define q_1n, q_const


In [ ]:
check_array("q_1n", q_1n, "4b509e7cfa2a6b2b", (2,))
check_array("q_const", q_const, "38833069b5a1259f", (2,))
print("ok  Exercise 7")


## Submitting

Choose Kernel, then Restart and Run All. Every check cell must print `ok`. Upload the `.ipynb` file on Canvas.
